# Tutorial - Train Classifier

In [1]:
# Suppress TensorFlow and MediaPipe logs
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow logs
os.environ['GLOG_minloglevel'] = '3'      # Suppress MediaPipe logs

import string 
import sys
import pandas as pd
from typing import Dict, List
from pathlib import Path

project_root = Path.cwd().parent
video_base_path = project_root / "data" / "youtube"
data_root = project_root / "experiments" / "exp1" / "data"

sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"data_root: {data_root}")

Project root: /Users/pmui/dev/alex/alexpose
data_root: /Users/pmui/dev/alex/alexpose/experiments/exp1/data


## 1.  Iterate through conditions

In [ ]:
import contextlib

# Set environment variables to suppress logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow logs
os.environ['GLOG_minloglevel'] = '3'      # Suppress MediaPipe/glog logs
os.environ['GLOG_logtostderr'] = '0'      # Don't log to stderr
os.environ['GLOG_stderrthreshold'] = '3'  # Only FATAL to stderr

# Additional suppression for absl logging (used by TensorFlow)
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_VMODULE'] = 'inference_feedback_manager=0,gl_context=0'

@contextlib.contextmanager
def suppress_stderr_fd():
    """Suppress stderr at file descriptor level to block C++ logs."""
    stderr_fd = sys.stderr.fileno()
    with open(os.devnull, 'w') as devnull:
        old_stderr = os.dup(stderr_fd)
        os.dup2(devnull.fileno(), stderr_fd)
        try:
            yield
        finally:
            os.dup2(old_stderr, stderr_fd)
            os.close(old_stderr)

with suppress_stderr_fd():
    from ambient.gavd import GaitDataProcessor, GAVDDataLoader, PoseDataConverter
    from ambient.pose.pose_estimators import OpenPoseEstimator
    from ambient.pose.keypoint_extractor import SequenceKeypointExtractor
    from ambient.pose.joint_angles import get_joint_angles
    from ambient.utils.csv_parser import parse_csv_with_dicts

from ambient.pose.pose_config import configure_pose_environment
configure_pose_environment()

import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)




In [3]:
# ignore any "dot" files
condition_paths = [
    p for p in Path(data_root).iterdir() 
    if p.is_dir() and p.name[0] in string.ascii_letters
]

In [4]:
[str(cp) for cp in condition_paths]

['/Users/pmui/dev/alex/alexpose/experiments/exp1/data/stroke',
 '/Users/pmui/dev/alex/alexpose/experiments/exp1/data/parkinsons']

In [5]:
gavd_loader = GAVDDataLoader()

In [6]:
for condition_path in condition_paths:
    print(f"==> condition_path: {condition_path}", flush=True)
    try:

        for csv_path in Path(condition_path).glob("*.csv"):
            print(f"\t>> {csv_path}", flush=True)
            df = gavd_loader.load_gavd_data(str(csv_path))
            sequences = gavd_loader.organize_by_sequence(df)

            extractor = SequenceKeypointExtractor()
            for seq_id in sequences:
                sequence_df = sequences[seq_id]
                keypoints_array = extractor.extract_from_sequence(
                    sequence_df,
                    video_base_path=video_base_path
                )

                joint_angles = get_joint_angles(
                    keypoints_array=keypoints_array,
                    keypoint_format="BLAZEPOSE_33",
                    fps=30.0,
                    confidence_threshold=0.3
                )

                
                # Validate before accessing frames
                if len(joint_angles.frames) == 0:
                    print(f"WARNING: No joint angle frames computed (keypoints: {len(keypoints_array)})", flush=True)
                    continue
                
                if len(joint_angles.frames[0].angles) == 0:
                    print(f"WARNING: First frame has no joint angles (frames: {len(joint_angles.frames)})", flush=True)
                    continue
                
                print(f"Average joint angles for {len(joint_angles.frames)} frames:", flush=True)
                for joint_name in joint_angles.frames[0].angles.keys():
                    avg_joint_angle = joint_angles.get_statistics(joint_name=joint_name)["mean"]
                    print(f"\t<joint angle> of {joint_name}: {avg_joint_angle}", flush=True)
    except Exception as e:
        print(f"Cannot process: {condition_path}: {e}")

==> condition_path: /Users/pmui/dev/alex/alexpose/experiments/exp1/data/stroke
	>> /Users/pmui/dev/alex/alexpose/experiments/exp1/data/stroke/cljo8c0sw005w3n6l9ulr2eg2.csv


2026-01-19 15:30:52.748 | INFO     | ambient.utils.youtube_cache:_prepare_download_list:232 - Already cached (skipping): /Users/pmui/dev/alex/alexpose/data/youtube/5gpoegYv1hs.mp4
2026-01-19 15:30:52.751 | DEBUG    | ambient.pose.mediapipe_singleton:__init__:50 - MediaPipe singleton initialized
I0000 00:00:1768865452.839503 6244000 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2 Max
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1768865452.888708 6244004 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1768865452.897334 6244006 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2026-01-19 15:30:52.898 | DEBUG    | ambient.pose.mediapipe_singleton:get_landmarker:95 - New landmarker created successfully
W0000 00:00:1768865453.112243 6244009 land

Average joint angles for 1 frames:
	<joint angle> of left_hip: 179.63620790928155
	<joint angle> of left_knee: 177.86111558971913
	<joint angle> of right_hip: 178.45677720374502
	<joint angle> of right_knee: 171.13125107270633
	<joint angle> of left_ankle: 154.63993878175972
	<joint angle> of right_ankle: 175.60711587004388
	>> /Users/pmui/dev/alex/alexpose/experiments/exp1/data/stroke/cljr5hwxc000f3n6lof5w9tyt.csv


2026-01-19 15:30:53.118 | INFO     | ambient.utils.youtube_cache:_prepare_download_list:232 - Already cached (skipping): /Users/pmui/dev/alex/alexpose/data/youtube/8mTHlAIdea0.mp4
I0000 00:00:1768865453.173592 6244075 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2 Max
W0000 00:00:1768865453.223028 6244077 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1768865453.231426 6244079 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2026-01-19 15:30:53.231 | DEBUG    | ambient.pose.mediapipe_singleton:get_landmarker:95 - New landmarker created successfully


==> condition_path: /Users/pmui/dev/alex/alexpose/experiments/exp1/data/parkinsons
	>> /Users/pmui/dev/alex/alexpose/experiments/exp1/data/parkinsons/cljnz6vg1000s3n6lxxknbe72.csv


2026-01-19 15:30:53.486 | INFO     | ambient.utils.youtube_cache:_prepare_download_list:232 - Already cached (skipping): /Users/pmui/dev/alex/alexpose/data/youtube/_Wn9oYGpRdM.mp4
I0000 00:00:1768865453.549014 6244147 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2 Max
W0000 00:00:1768865453.598961 6244148 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1768865453.608398 6244158 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2026-01-19 15:30:53.608 | DEBUG    | ambient.pose.mediapipe_singleton:get_landmarker:95 - New landmarker created successfully


Average joint angles for 1 frames:
	<joint angle> of left_hip: 175.26293139853198
	<joint angle> of left_knee: 178.69702616460262
	<joint angle> of right_hip: 171.89681966891965
	<joint angle> of right_knee: 174.58211076997634
	<joint angle> of left_ankle: 121.22145753182959
	<joint angle> of right_ankle: 177.08811822112304
	>> /Users/pmui/dev/alex/alexpose/experiments/exp1/data/parkinsons/cljnz5sb1000o3n6lntosswwz.csv


2026-01-19 15:30:53.851 | INFO     | ambient.utils.youtube_cache:_prepare_download_list:232 - Already cached (skipping): /Users/pmui/dev/alex/alexpose/data/youtube/_Wn9oYGpRdM.mp4
I0000 00:00:1768865453.906968 6244219 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2 Max
W0000 00:00:1768865453.955580 6244220 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1768865453.963904 6244222 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2026-01-19 15:30:53.964 | DEBUG    | ambient.pose.mediapipe_singleton:get_landmarker:95 - New landmarker created successfully


Average joint angles for 1 frames:
	<joint angle> of left_hip: 170.61583123115255
	<joint angle> of left_knee: 163.20226576651493
	<joint angle> of right_hip: 165.66363292421974
	<joint angle> of right_knee: 172.09448891537485
	<joint angle> of left_ankle: 168.39682073335854
	<joint angle> of right_ankle: 143.2011805503621
